# CEI as a root-cause ranking head in RCAEval: a null result

This notebook reloads the saved results of the pre-registered study in `../PLAN.md` and rebuilds every table in the README. Nothing is recomputed from raw telemetry here; `experiments/` has the code that produced the files.

**Verdict.** On all 735 RCAEval cases the Centrality-Entropy Index head ranked the true root-cause service lower than RCAEval's shipped PageRank head on both PC and Granger graphs, and the CEI × PageRank hybrid lost on PC graphs and tied on Granger graphs. Under the rule committed before the runs, no CEI pull request follows.

In [ ]:
import json
import pandas as pd
pd.set_option('display.width', 160)
R = '../results/'

## 1. Data path check
BARO on RE2-TT reproduces the Avg@5 published in RCAEval's README for every fault.

In [ ]:
pd.read_csv(R + 'baro_reproduction.csv')

## 2. Accuracy of every arm (service level, all 735 cases)

In [ ]:
am = pd.read_csv(R + 'arm_metrics.csv')
am[am.group == 'all'][['arm', 'n', 'AC@1', 'AC@3', 'Avg@5', 'lift_Avg@5']].round(3)

## 3. Pre-registered tests
Paired over cases; Wilcoxon signed-rank with Holm correction across the four tests; 95% bootstrap CIs (10,000 resamples). A win needs the CI above zero.

In [ ]:
pd.read_csv(R + 'primary_tests.csv')[['a', 'b', 'mean_a', 'mean_b', 'diff', 'ci_lo', 'ci_hi', 'p_holm', 'verdict']].round(4)

In [ ]:
json.load(open(R + 'summary.json'))['decision']

## 4. Robustness of the verdicts

In [ ]:
pd.read_csv(R + 'sensitivity_tests.csv')[['subset', 'a', 'n', 'diff', 'ci_lo', 'ci_hi', 'verdict']].round(4)

## 5. Which part of CEI failed (secondary, uncorrected) and exploratory checks
The entropy term alone ranks below random; the centrality term alone beats full CEI. The exploratory rows were added after the primary results and need their own pre-registered test.

In [ ]:
sec = pd.read_csv(R + 'secondary_tests.csv')
sec[['a', 'b', 'diff', 'ci_lo', 'ci_hi', 'p', 'verdict']].round(4)

In [ ]:
pd.read_csv(R + 'exploratory_tests.csv')[['a', 'b', 'diff', 'ci_lo', 'ci_hi', 'p', 'verdict']].round(4)

## 6. By system and suite

In [ ]:
keep = ['pc:pagerank', 'pc:cei', 'granger:pagerank', 'granger:cei', 'granger:cei_pagerank', 'baro', 'random']
t = am[am.arm.isin(keep) & am.group.str.match('system|suite')].pivot(index='group', columns='arm', values='Avg@5')[keep]
t.round(3)

## 7. Graph building
PC did not finish within 20 minutes on any RE2-TT case; those cases get the benchmark's fallback for every head.

In [ ]:
gb = pd.read_csv(R + 'graph_builds.csv')
gb.groupby(['kind', 'dataset', 'status']).size().unstack(fill_value=0)

## 8. Side findings reported upstream
* The shipped random-walk head ranks identically on PC and Granger graphs (it never reads the graph).
* The Hugging Face release differs from the Zenodo cases `main.py` reads (see `../evidence/source_compare.json`).

In [ ]:
json.load(open(R + 'randomwalk_graph_independence.json'))

In [ ]:
cmp = json.load(open('../evidence/source_compare.json')); cmp.pop('_errors', None)
df = pd.DataFrame(cmp).T
df['suite'] = df.index.str[:5]
df.groupby('suite').identical_input.value_counts().unstack(fill_value=0)